# Code LRM ckpt 训练（Colab）

用**真实公开 code 数据集**训三个 latent 推理 ckpt：**CoLaR / LT-Tuning / Latent-SFT**。
数据 1488 train / 165 val（CRUXEval + LiveCodeBench + MBPP）。

训练配方不在本 notebook 里 —— 直接调 mirror 的 `handoff/*.sh`，
那是**唯一的配方源头**（与服务器跑的完全同一套，已本地验证）：
https://github.com/ruijiezh67/LRM_colab_tasks/tree/main/handoff

---

## ⚠️ 必须分三段跑，段间重启运行时

三个平台的 `transformers` 版本**互斥**：

| 平台 | transformers | 其它 |
|---|---|---|
| CoLaR | **4.45.2** | lightning 2.5.1 |
| LT-Tuning | **4.55.4** | torch 2.7.1 / deepspeed 0.18.3 |
| Latent-SFT | **4.51.1** | deepspeed 0.17.0 |

一个 runtime 装不下三套。**每段开始前先 `Runtime → Restart session`，再从「公共 SETUP」重跑。**

## 每段的流程

`公共 SETUP` → `训练` → `loss 检查` → `存出 ckpt`

⚠️ **存出那一格别跳过。** Colab 断连产物就没了。

## 大概时长（A100）

CoLaR ~1–2h · LT-Tuning ~1h · Latent-SFT ~3–5h


---
# 公共 SETUP

**每次重启运行时后都要先跑这一格。**

In [ ]:
# ── 公共 SETUP（每次重启运行时后都要先跑这一格）────────────────
import os, subprocess, shutil, glob, time
from pathlib import Path

print("GPU:"); os.system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

WORK = "/content/work"           # Colab 本地盘, 训练产物落这里
REPO = "/content/LRM_colab_tasks"
os.makedirs(WORK, exist_ok=True)

if not os.path.isdir(REPO):
    os.system(f"git clone -q https://github.com/ruijiezh67/LRM_colab_tasks.git {REPO}")
print("mirror HEAD:", subprocess.run(
    ["git","-C",REPO,"log","-1","--format=%h %s"], capture_output=True, text=True).stdout.strip())

# 交接脚本自带起飞前自检(python/venv/卡/上游源/数据/磁盘), 保持开着。
# Colab 通常有 ~78GB 可用, 能过磁盘那项(要求 >60GB)。
# 万一 Colab 分到的盘偏小而其它项都 OK, 再改成 PREFLIGHT="0" 跳过。
os.environ["WORK"] = WORK
os.environ["PREFLIGHT"] = "1"
os.system("df -h /content | tail -1")
print("
WORK =", WORK)
print("上游代码 vendored:", os.path.isdir(f"{REPO}/upstream/colar"))
print("训练数据就位   :", os.path.isfile(f"{REPO}/code_real_ladder/manifest.json"))


---
# 段 A · CoLaR

底座 Llama-3.2-1B + warm-start CoLaR-GSM　→　产物 `colar_code_cruxreal.ckpt`

> 跑之前确认已重启运行时并跑过「公共 SETUP」。

In [ ]:
# ── 训练 CoLaR ──────────────────────────────────────────────
# 直接调 handoff 脚本(唯一的训练配方源头), Colab 只负责触发
import os, subprocess, sys
t0 = time.time()
p = subprocess.Popen(
    ["bash", "train_all_code.sh"],
    cwd=f"{REPO}/handoff",
    env={**os.environ, "ONLY": "colar", "PARALLEL": "0", "WORK": WORK, "CN": "0"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:        # 实时打印, 不要等跑完才出日志
    sys.stdout.write(line); sys.stdout.flush()
p.wait()
print(f"
rc={p.returncode}  用时 {(time.time()-t0)/60:.1f} 分")
if p.returncode: print("失败 → 看上面日志; 完整日志在", f"{WORK}/run_logs/")


In [ ]:
# ── loss 收敛检查（golden rule: 必须画出来看）──────────────────
import re, glob, matplotlib.pyplot as plt
logs = sorted(glob.glob(f"{WORK}/run_logs/*.log")) + sorted(glob.glob(f"{WORK}/run_logs/*.stdout.log"))
print("日志:", [Path(x).name for x in logs])
for lg in logs:
    txt = open(lg, encoding="utf-8", errors="ignore").read()
    ls = [float(x) for x in re.findall(r"(?:'loss'|train_loss|loss)[=:'\s]+([0-9]+\.[0-9]+)", txt)]
    ls = [x for x in ls if x < 1e4]
    if len(ls) >= 2:
        plt.figure(figsize=(6,3)); plt.plot(ls, marker="."); plt.title(Path(lg).name+" · loss")
        plt.xlabel("log step"); plt.ylabel("loss"); plt.grid(alpha=.3); plt.show()
        print(f"{Path(lg).name}: {ls[0]:.3f} -> {ls[-1]:.3f} ({len(ls)} 点)",
              "✅ 收敛" if ls[-1] < ls[0] else "⚠ 没降, 查 lr/数据/配方")


In [ ]:
# ── 把 ckpt 存出去（铁律: 不把产物留在 Colab, 断连就没了）─────────
# 产物路径见 handoff/README.md 第 3 节
import os, glob, shutil
from pathlib import Path

cands = ([f"{WORK}/colar_code_cruxreal.ckpt"]
         + glob.glob(f"{WORK}/lt_code_out/qwen_code")
         + glob.glob(f"{WORK}/Latent-SFT/output/stage2_results/code/checkpoint-*/hf"))
found = [c for c in cands if os.path.exists(c)]
for c in found:
    sz = (os.path.getsize(c) if os.path.isfile(c)
          else sum(f.stat().st_size for f in Path(c).rglob("*") if f.is_file()))
    print(f"{sz/1048576:8.1f} MB  {c}")
if not found: print("没找到产物 —— 训练可能没成功, 先看上面的 rc 和日志")

# 两条路, 按体积选:
#
# A) 小产物(几百 MB 以内) → 直接下载到本机, 再挪到 E 盘
#    from google.colab import files
#    files.download(f"{WORK}/colar_code_cruxreal.ckpt")
#
# B) 大产物(GB 级, LT/LSFT 的整个模型目录) → files.download 不现实, 传 HF
#    from huggingface_hub import HfApi, login
#    login(token="hf_...")                      # 别把 token 写进 notebook 提交
#    HfApi().upload_folder(folder_path=f"{WORK}/lt_code_out/qwen_code",
#                          repo_id="rjz123/lt-code-cruxreal-qwen15b",
#                          repo_type="model", private=False)


---
# 段 B · LT-Tuning

底座 Qwen2.5-1.5B-Instruct，三阶段课程　→　产物 `lt_code_out/qwen_code/`

> ⚠️ **先 `Runtime → Restart session`**（transformers 要从 4.45.2 换到 4.55.4），再跑「公共 SETUP」。

In [ ]:
# ── 训练 LT-Tuning ──────────────────────────────────────────────
# 直接调 handoff 脚本(唯一的训练配方源头), Colab 只负责触发
import os, subprocess, sys
t0 = time.time()
p = subprocess.Popen(
    ["bash", "train_all_code.sh"],
    cwd=f"{REPO}/handoff",
    env={**os.environ, "ONLY": "lt", "PARALLEL": "0", "WORK": WORK, "CN": "0"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:        # 实时打印, 不要等跑完才出日志
    sys.stdout.write(line); sys.stdout.flush()
p.wait()
print(f"
rc={p.returncode}  用时 {(time.time()-t0)/60:.1f} 分")
if p.returncode: print("失败 → 看上面日志; 完整日志在", f"{WORK}/run_logs/")


In [ ]:
# ── loss 收敛检查（golden rule: 必须画出来看）──────────────────
import re, glob, matplotlib.pyplot as plt
logs = sorted(glob.glob(f"{WORK}/run_logs/*.log")) + sorted(glob.glob(f"{WORK}/run_logs/*.stdout.log"))
print("日志:", [Path(x).name for x in logs])
for lg in logs:
    txt = open(lg, encoding="utf-8", errors="ignore").read()
    ls = [float(x) for x in re.findall(r"(?:'loss'|train_loss|loss)[=:'\s]+([0-9]+\.[0-9]+)", txt)]
    ls = [x for x in ls if x < 1e4]
    if len(ls) >= 2:
        plt.figure(figsize=(6,3)); plt.plot(ls, marker="."); plt.title(Path(lg).name+" · loss")
        plt.xlabel("log step"); plt.ylabel("loss"); plt.grid(alpha=.3); plt.show()
        print(f"{Path(lg).name}: {ls[0]:.3f} -> {ls[-1]:.3f} ({len(ls)} 点)",
              "✅ 收敛" if ls[-1] < ls[0] else "⚠ 没降, 查 lr/数据/配方")


In [ ]:
# ── 把 ckpt 存出去（铁律: 不把产物留在 Colab, 断连就没了）─────────
# 产物路径见 handoff/README.md 第 3 节
import os, glob, shutil
from pathlib import Path

cands = ([f"{WORK}/colar_code_cruxreal.ckpt"]
         + glob.glob(f"{WORK}/lt_code_out/qwen_code")
         + glob.glob(f"{WORK}/Latent-SFT/output/stage2_results/code/checkpoint-*/hf"))
found = [c for c in cands if os.path.exists(c)]
for c in found:
    sz = (os.path.getsize(c) if os.path.isfile(c)
          else sum(f.stat().st_size for f in Path(c).rglob("*") if f.is_file()))
    print(f"{sz/1048576:8.1f} MB  {c}")
if not found: print("没找到产物 —— 训练可能没成功, 先看上面的 rc 和日志")

# 两条路, 按体积选:
#
# A) 小产物(几百 MB 以内) → 直接下载到本机, 再挪到 E 盘
#    from google.colab import files
#    files.download(f"{WORK}/colar_code_cruxreal.ckpt")
#
# B) 大产物(GB 级, LT/LSFT 的整个模型目录) → files.download 不现实, 传 HF
#    from huggingface_hub import HfApi, login
#    login(token="hf_...")                      # 别把 token 写进 notebook 提交
#    HfApi().upload_folder(folder_path=f"{WORK}/lt_code_out/qwen_code",
#                          repo_id="rjz123/lt-code-cruxreal-qwen15b",
#                          repo_type="model", private=False)


---
# 段 C · Latent-SFT

底座 Llama-3.2-1B-Instruct，6 步管线　→　产物 `Latent-SFT/output/stage2_results/code/checkpoint-*/hf`

> ⚠️ **先 `Runtime → Restart session`**（transformers 换到 4.51.1），再跑「公共 SETUP」。

In [ ]:
# ── 训练 Latent-SFT ──────────────────────────────────────────────
# 直接调 handoff 脚本(唯一的训练配方源头), Colab 只负责触发
import os, subprocess, sys
t0 = time.time()
p = subprocess.Popen(
    ["bash", "train_all_code.sh"],
    cwd=f"{REPO}/handoff",
    env={**os.environ, "ONLY": "lsft", "PARALLEL": "0", "WORK": WORK, "CN": "0"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:        # 实时打印, 不要等跑完才出日志
    sys.stdout.write(line); sys.stdout.flush()
p.wait()
print(f"
rc={p.returncode}  用时 {(time.time()-t0)/60:.1f} 分")
if p.returncode: print("失败 → 看上面日志; 完整日志在", f"{WORK}/run_logs/")


In [ ]:
# ── loss 收敛检查（golden rule: 必须画出来看）──────────────────
import re, glob, matplotlib.pyplot as plt
logs = sorted(glob.glob(f"{WORK}/run_logs/*.log")) + sorted(glob.glob(f"{WORK}/run_logs/*.stdout.log"))
print("日志:", [Path(x).name for x in logs])
for lg in logs:
    txt = open(lg, encoding="utf-8", errors="ignore").read()
    ls = [float(x) for x in re.findall(r"(?:'loss'|train_loss|loss)[=:'\s]+([0-9]+\.[0-9]+)", txt)]
    ls = [x for x in ls if x < 1e4]
    if len(ls) >= 2:
        plt.figure(figsize=(6,3)); plt.plot(ls, marker="."); plt.title(Path(lg).name+" · loss")
        plt.xlabel("log step"); plt.ylabel("loss"); plt.grid(alpha=.3); plt.show()
        print(f"{Path(lg).name}: {ls[0]:.3f} -> {ls[-1]:.3f} ({len(ls)} 点)",
              "✅ 收敛" if ls[-1] < ls[0] else "⚠ 没降, 查 lr/数据/配方")


In [ ]:
# ── 把 ckpt 存出去（铁律: 不把产物留在 Colab, 断连就没了）─────────
# 产物路径见 handoff/README.md 第 3 节
import os, glob, shutil
from pathlib import Path

cands = ([f"{WORK}/colar_code_cruxreal.ckpt"]
         + glob.glob(f"{WORK}/lt_code_out/qwen_code")
         + glob.glob(f"{WORK}/Latent-SFT/output/stage2_results/code/checkpoint-*/hf"))
found = [c for c in cands if os.path.exists(c)]
for c in found:
    sz = (os.path.getsize(c) if os.path.isfile(c)
          else sum(f.stat().st_size for f in Path(c).rglob("*") if f.is_file()))
    print(f"{sz/1048576:8.1f} MB  {c}")
if not found: print("没找到产物 —— 训练可能没成功, 先看上面的 rc 和日志")

# 两条路, 按体积选:
#
# A) 小产物(几百 MB 以内) → 直接下载到本机, 再挪到 E 盘
#    from google.colab import files
#    files.download(f"{WORK}/colar_code_cruxreal.ckpt")
#
# B) 大产物(GB 级, LT/LSFT 的整个模型目录) → files.download 不现实, 传 HF
#    from huggingface_hub import HfApi, login
#    login(token="hf_...")                      # 别把 token 写进 notebook 提交
#    HfApi().upload_folder(folder_path=f"{WORK}/lt_code_out/qwen_code",
#                          repo_id="rjz123/lt-code-cruxreal-qwen15b",
#                          repo_type="model", private=False)


---
# 出问题时

1. **先看 `rc=` 和实时日志**（脚本自带起飞前自检，不满足会当场停下并给修复命令）
2. 完整日志：`{WORK}/run_logs/*.stdout.log` 的**最后 25 行**
3. 上游代码与数据都随 mirror 分发，**训练全程不需要访问 GitHub**；
   唯一联网点是从 HuggingFace 下三个底座权重
4. 小规模排错：把训练格的 `ONLY` 那行旁边加 `"VERIFY": "1"`（200 行，不出正式 ckpt）

说明文档：`handoff/README.md`（一份，含配方对照与已知边界）
